In [1]:
import sys
sys.path.append("../")
from preprocessing.create_train_test_dicts import build_splits

ds = build_splits()
train_split = ds['train_numeric']
val_split = ds['val_numeric']
test_split = ds['test_numeric']


In [2]:
from preprocessing.cnn_pipeline import CNNParamSearch, plot_roc_curves, plot_pr_curves, plot_confusion_matrix

search = CNNParamSearch(
    train_dict=train_split,
    val_dict=val_split,
    lrs=[1e-3, 3e-4],
    weight_decays=[1e-4, 5e-5],
    batch_size=128,
    num_epochs=10,
)

results = search.run()


ModuleNotFoundError: No module named 'torch'

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

pivot_f1 = df.pivot_table(index="lr", columns="weight_decay", values="val_f1_weighted")
plt.figure(figsize=(6, 4))
sns.heatmap(pivot_f1, annot=True, cmap="magma")
plt.title("CNN Validation F1-weighted")
plt.xlabel("weight_decay")
plt.ylabel("lr")
plt.tight_layout()
plt.show()


In [ ]:
model, test_logits, test_labels, class_names = search.train_best_model(
    results,
    test_dict=test_split,
    metric_name="val_f1_weighted",
)

y_pred = test_logits.argmax(axis=1)
plot_confusion_matrix(test_labels, y_pred, class_names)
plot_confusion_matrix(test_labels, y_pred, class_names, normalize=True)
plot_roc_curves(test_labels, test_logits, class_names)
plot_pr_curves(test_labels, test_logits, class_names)
